# 🗓 Calendar Agent — Lightning AI Studio LoRA Training

Lightning AI Studio 환경에서 `calendar-agent` (0.5B / 1.5B) 모델을 학습하고 평가하는 노트북입니다.

## 💡 실행 전 준비
1. **GPU 확인**: 우측 상단에서 `A10G` 또는 `T4` GPU가 켜져 있는지 확인하세요. (학습하지 않을 때는 CPU 모드로 전환하여 크레딧 절약)
2. **Hugging Face 토큰**: 데이터셋 연동 및 모델 백업을 위해 Hugging Face `Write` 권한 토큰이 필요합니다.

## 0. GPU 작동 확인
A10G 또는 Tesla T4 등이 정상적으로 인식되는지 확인합니다.

In [ ]:
!nvidia-smi

## 1. 경로 설정 및 이동
Lightning AI Studio의 기본 작업 디렉토리는 `/teamspace/studios/this_studio` 입니다. 프로젝트 폴더로 이동합니다.

In [ ]:
import os

# 깃클론을 하지 않고 직접 드래그앤드롭으로 코드를 올렸을 때를 대비한 경로 체크
default_path = '/teamspace/studios/this_studio'
project_name = 'calendar-agent'

if os.path.exists(os.path.join(default_path, project_name)):
    %cd {os.path.join(default_path, project_name)}
else:
    # 현재 디렉토리가 이미 프로젝트 폴더 안인지 확인
    if os.path.basename(os.getcwd()) == project_name:
        print(f"이미 {project_name} 디렉토리에 있습니다.")
    else:
        print(f"[주의] {project_name} 폴더를 찾을 수 없습니다. 현재 경로: {os.getcwd()}")

!pwd

## 2. 의존성 설치 및 캐시 클린업
프로젝트를 개발용 패키지로 설치하고, 학습에 불필요한 패키지를 정리합니다.

In [ ]:
# 학습 의존성 패키지 설치 (Lightning은 %pip 권장 — 커널 환경에 정확히 설치)
%pip install -q -e ".[train]"
# PEFT 충돌을 방지하기 위해 불필요한 torchao 제거
%pip uninstall torchao -y -q

import os, torch
os.environ["WANDB_DISABLED"] = "true"
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "ngpu:", torch.cuda.device_count())

## 3. Hugging Face 동기화 (생성된 Git 저장소에 데이터가 이미 포함되어 있어 생략합니다)

> [!NOTE]
> 이 프로젝트는 데이터셋이 이미 GitHub 저장소(`data/`)에 완전히 포함되어 추적되고 있습니다.
> 따라서 Hugging Face에서 데이터를 다운로드받는 단계는 완전히 생략하셔도 괜찮습니다.

In [ ]:
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# # Hugging Face 로그인 (발급받은 Write 권한 토큰을 입력하세요)
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# !huggingface-cli login

In [ ]:
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# USER_NAME = "your-username"  # 본인의 Hugging Face 사용자명으로 변경하세요
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# DATASET_REPO = f"{USER_NAME}/calendar-agent-dataset"
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# 
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# # Hugging Face에서 데이터셋 다운로드
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# !huggingface-cli download {DATASET_REPO} --local-dir ./data --repo-type dataset
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# 
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# # 다운로드한 데이터셋 레코드 수 확인
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
# for p in ["data/processed/train.jsonl", "data/processed/val.jsonl", "data/eval/golden.jsonl"]:
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
#     if os.path.exists(p):
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
#         n = sum(1 for _ in open(p, encoding="utf-8"))
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
#         print(f"{p}: {n} records")
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
#     else:
# GitHub에 이미 데이터가 포함되어 있으므로 이 셀은 실행하지 않습니다.
#         print(f"{p} 파일이 존재하지 않습니다.")

## 4. 모델 학습 실행
선택한 모델 설정을 로드하여 학습을 진행합니다. 아래 `CONFIG` 한 줄만 바꾸면 됩니다.

* **Qwen3-0.6B (r29 A/B, 기본)**: `configs/train_lightning_qwen3_0_6b.yaml`
* **0.5B**: `configs/train.yaml`
* **1.5B**: `configs/train_lightning_1_5b.yaml`

In [ ]:
CONFIG = 'configs/train_lightning_qwen3_0_6b.yaml'  # L4 GPU Qwen3-0.6B A/B (r29). 1.5B로 되돌리려면 'configs/train_lightning_1_5b.yaml'
print('학습 config:', CONFIG)

import torch
num_gpus = torch.cuda.device_count()

if num_gpus > 1:
    # GPU가 2개 이상일 때는 DDP(torchrun) 사용
    print(f"멀티 GPU {num_gpus}개 감지: torchrun으로 분산 학습을 시작합니다.")
    !torchrun --nproc_per_node={num_gpus} scripts/train_lora.py --config {CONFIG}
else:
    # 싱글 GPU(A10G 1개 등)일 때는 단일 프로세스 실행
    print("싱글 GPU 감지: 단일 프로세스로 학습을 시작합니다.")
    !python scripts/train_lora.py --config {CONFIG}

## 5. 모델 병합 및 평가
학습이 끝난 LoRA 어댑터를 베이스 모델과 병합(Merge)한 뒤, 골든셋으로 성능을 평가합니다.

In [ ]:
import yaml, os

# 설정 파일 파싱
_cfg = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))

BASE = _mcfg['hf_id']                          # 베이스 모델명 자동 검출
LORA_DIR = _cfg['output_dir']                  # LoRA 가중치 저장 폴더
NAME = os.path.basename(LORA_DIR)              # 저장 폴더명 (예: r12-qwen0.5b)
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON = f'logs/eval_{NAME}.json'
GOLDEN = _cfg.get('eval_golden', 'data/eval/golden.jsonl')

print(f'베이스 모델: {BASE}')
print(f'LoRA 가중치 경로: {LORA_DIR}')
print(f'병합 대상 경로: {MERGED_DIR}')

# 1. LoRA 모델과 베이스 모델 병합
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}

# 2. 골든셋 평가
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON}

print(f'\n--- 평가 결과 ({EVAL_JSON}) ---')
if os.path.exists(EVAL_JSON):
    print(open(EVAL_JSON, encoding='utf-8').read())
else:
    print("평가 결과 파일을 찾을 수 없습니다.")

## 6. 학습 결과 백업 안내

> [!NOTE]
> 학습이 완료된 LoRA 모델 가중치는 Lightning AI Studio 내부 스토리지(`models/lora/r12-qwen1.5b-lightning`)에 안전하게 유지됩니다.
> 로컬 PC로 가져오고 싶으실 때만 왼쪽 파일 탐색기에서 해당 폴더를 마우스 우클릭하여 **[Download]**를 진행해 주세요.

In [ ]:
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# MODEL_REPO = f"{USER_NAME}/calendar-agent-lora"
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# 
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# print(f"Hugging Face 모델 저장소: {MODEL_REPO}")
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# print(f"업로드할 폴더: {LORA_DIR}")
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# 
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# # 1. Hugging Face에 모델 저장소 생성 (이미 존재하면 무시, 비공개 생성)
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# from huggingface_hub import HfApi
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# api = HfApi()
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# api.create_repo(repo_id=MODEL_REPO, repo_type="model", private=True, exist_ok=True)
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# 
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# # 2. LoRA 어댑터 폴더 업로드
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# !huggingface-cli upload {MODEL_REPO} {LORA_DIR} . --repo-type model
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# 
# 학습 모델은 스튜디오에 보존되므로 이 셀은 실행하지 않습니다.
# print("Hugging Face 모델 업로드 완료!")